# Neural Networks with PyTorch — XOR Problem (SOLUTION)

## Step 1: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', device)

## Step 2: Dataset

In [ ]:
X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32).to(device)
y = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32).to(device)
print('X:', X)
print('y:', y)

## Step 3: Define the Network with nn.Module

`nn.Module` is PyTorch's base class. Define layers in `__init__`, the forward pass in `forward()`.

In [ ]:
class XORNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 4)
        self.output = nn.Linear(4, 1)

    def forward(self, x):
        x = torch.sigmoid(self.hidden(x))
        return torch.sigmoid(self.output(x))

model = XORNet().to(device)
print(model)
print('Parameters:', sum(p.numel() for p in model.parameters()))

## Step 4: Loss Function and Optimizer

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

## Step 5: Training Loop

5-step pattern every iteration: forward → loss → zero_grad → backward → step

In [ ]:
losses = []
for epoch in range(10000):
    out  = model(X)
    loss = criterion(out, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if epoch % 1000 == 0:
        print(f'Epoch {epoch:5d} | Loss: {loss.item():.4f}')

## Step 6: Evaluate

In [ ]:
model.eval()
with torch.no_grad():
    preds = torch.round(model(X))
print('True:     ', y.cpu().flatten())
print('Predicted:', preds.cpu().flatten())

## Step 7: Inspect Weights

In [ ]:
print(model.state_dict())

## Step 8: Visualize the Network Architecture

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def draw_network(layer_sizes, layer_labels=None, title='', figsize=(9, 5)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.2, 1.2)
    ax.axis('off')
    if title:
        ax.set_title(title, fontsize=13, fontweight='bold')
    n = len(layer_sizes)
    xs = np.linspace(0.1, 0.9, n)
    max_n = max(layer_sizes)
    gap = min(0.15, 0.85 / max(max_n, 2))
    colors = ['#74b9ff', '#a29bfe', '#fd79a8', '#55efc4', '#ffeaa7']
    ys_all = []
    for sz in layer_sizes:
        ys = [0.5 + (i - (sz - 1) / 2) * gap for i in range(sz)]
        ys_all.append(ys)
    for li in range(n - 1):
        for ya in ys_all[li]:
            for yb in ys_all[li + 1]:
                ax.plot([xs[li], xs[li + 1]], [ya, yb], color='#dfe6e9', lw=0.8, zorder=1)
    for li, (x, ys) in enumerate(zip(xs, ys_all)):
        for y in ys:
            c = plt.Circle((x, y), 0.038, fc=colors[li % len(colors)], ec='#2d3436', lw=1.2, zorder=5)
            ax.add_patch(c)
        lab = layer_labels[li] if layer_labels else str(layer_sizes[li])
        ax.text(x, min(ys) - 0.1, lab, ha='center', fontsize=9, color='#636e72')
    plt.tight_layout()
    plt.show()

draw_network(
    [2, 4, 1],
    layer_labels=['Input\n(2)', 'Hidden\n(4)', 'Output\n(1)'],
    title='XOR Network: 2 → 4 → 1'
)

## Step 9: Plot Training Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses, color='#74b9ff', linewidth=1.5)
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Training Loss — PyTorch XOR')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Bonus: Adam optimizer

In [ ]:
model2 = XORNet().to(device)
opt2   = optim.Adam(model2.parameters(), lr=0.01)
adam_losses = []
for epoch in range(3000):
    out  = model2(X)
    loss = criterion(out, y)
    opt2.zero_grad()
    loss.backward()
    opt2.step()
    adam_losses.append(loss.item())
    if epoch % 500 == 0:
        print(f'Epoch {epoch:5d} | Loss: {loss.item():.4f}')

plt.figure(figsize=(8, 4))
plt.plot(losses[:3000], label='SGD', color='#74b9ff')
plt.plot(adam_losses,   label='Adam', color='#fd79a8')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('SGD vs Adam — first 3 000 epochs')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()